# Stage 3 (Refinement & Enrichment) — end-to-end

Exercises Stage 3 exactly as it runs in production: it is **not** a separate API
call, it fires automatically inside `PipelineService._run()` right after a job's
`final_status` becomes `"accepted"`. So this notebook drives it the same way
`playground/stage1/pipeline_end_to_end.ipynb` drives Stage 1/2 — real `POST
/pipeline/ingest` calls through the real production `Container` and a real SQLite
database — and then inspects the `EnrichedRecord` row that Stage 3 writes.

Two things are mocked for determinism, both node3's legitimacy check (as in the
Stage 1 notebook) and the entity-extraction LLM call — real text cleaning and
metadata enrichment run unmocked. `set_entity_response(...)` controls what the
mocked SLM returns for entity extraction; feed it malformed JSON to see the
retry-then-review failure path in section 4.

> **Heads up**: section 1 deletes and recreates `data/classiflow.db` every run, same
> as the Stage 1 notebook — start each run from a clean slate.

## 1 — App setup: the real Container, JWT auth

In [1]:
from pathlib import Path

from fastapi.testclient import TestClient
from sqlalchemy import select
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

import classiflow
from classiflow.api.app import create_app
from classiflow.database.base import Base
from classiflow.database.models import AllowedUser, EnrichedRecord, Job
from classiflow.injections.production import Container
from classiflow.services.auth import encode_token
from classiflow.settings import Settings

Settings.JWT_SECRET_KEY = "playground-secret-key-not-for-prod-use-only-demo"

# Anchor DATABASE_URL to the package location -- see pipeline_end_to_end.ipynb's
# section 1 for why a relative path breaks depending on the kernel's cwd.
_project_root = Path(classiflow.__file__).parents[2]
_db_path = _project_root / "data" / "classiflow.db"

for _stale in (_db_path, _db_path.with_suffix(".db-wal"), _db_path.with_suffix(".db-shm")):
    if _stale.exists():
        _stale.unlink()
print(f"reset database at {_db_path}")

Settings.DATABASE_URL = f"sqlite+aiosqlite:///{_db_path.as_posix()}"

container = Container()
container.wire(packages=["classiflow"])

engine = create_async_engine(Settings.DATABASE_URL, echo=False)
session_factory = async_sessionmaker(engine, expire_on_commit=False)


async def _create_tables() -> None:
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)


async def _seed_user(email: str) -> None:
    async with session_factory() as session:
        existing = await session.execute(select(AllowedUser).where(AllowedUser.email == email))
        if existing.scalar_one_or_none() is None:
            session.add(AllowedUser(email=email, is_active=True, is_blocked=False))
            await session.commit()


await _create_tables()
_EMAIL = "leonardo.heis@gmail.com"
await _seed_user(_EMAIL)

client = TestClient(create_app())
auth_headers = {"Authorization": f"Bearer {encode_token(_EMAIL)}"}

print(f"logged in as {_EMAIL}")
print(f"writing to {_db_path}")


c:\Users\leona\source\repos\Trabajo-Integrador\.venv\lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
c:\Users\leona\source\repos\Trabajo-Integrador\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\leona\source\repos\Trabajo-Integrador\src\classiflow\api\dependencies.py:215: DIWiringWarning: @inject is not required here
  def get_coordinator(


reset database at C:\Users\leona\source\repos\Trabajo-Integrador\data\classiflow.db
logged in as leonardo.heis@gmail.com
writing to C:\Users\leona\source\repos\Trabajo-Integrador\data\classiflow.db


## 2 — A real sample PDF, and the two mocks

`set_legitimacy(is_legitimate=True)` guarantees node3 accepts the document, so it
always reaches Stage 3 regardless of what the real SLM would decide.
`set_entity_response(json_text)` controls the entity extraction chain's raw LLM
response -- feed it valid JSON for the happy path, or something malformed to trigger
Stage 3's retry-then-review path in section 5.

Both mocks override the `Container`'s own provider
(`container.node3_content_chain.override(...)` /
`container.entity_extraction_chain.override(...)`) rather than monkeypatching a
module-level `get_llm_langchain` name -- the chain a node actually receives comes
from `injections/production.py`'s own import of `get_llm_langchain`, a *different*
name binding than `node3_content_validation.py`'s or `entity_extractor.py`'s own
`from ... import get_llm_langchain`, so patching one doesn't affect the other.
Overriding the provider directly is the reliable way to mock a real, fully-wired
`Container` in a notebook.

In [2]:
from pathlib import Path

import classiflow
from dependency_injector import providers
from classiflow.enrichment.prompts.entity_extraction import build_entity_extraction_chain
from classiflow.ingesta.llm_provider import MockLlm
from classiflow.ingesta.prompts import build_content_chain

_SAMPLES_DIR = Path(classiflow.__file__).parent / "playground" / "samples"
_SAMPLE_PDF = (_SAMPLES_DIR / "ordenanza_6801_1999.pdf").read_bytes()

_SLM_LEGITIMATE = '{"is_legitimate": true, "confidence": 0.92, "reasoning": "official doc"}'
_SLM_NOT_LEGITIMATE = '{"is_legitimate": false, "confidence": 0.9, "reasoning": "n/a"}'
_VALID_ENTITY_RESPONSE = (
    '{"doc_type_hint": "ordenanza", "number": "6801", "year": 1999, '
    '"issuing_body": "Concejo Municipal", "signatories": [], "article_count": 3}'
)


def set_legitimacy(*, is_legitimate: bool) -> None:
    response = _SLM_LEGITIMATE if is_legitimate else _SLM_NOT_LEGITIMATE
    container.node3_content_chain.override(
        providers.Object(build_content_chain(MockLlm(response=response)))
    )


def set_entity_response(json_text: str) -> None:
    container.entity_extraction_chain.override(
        providers.Object(build_entity_extraction_chain(MockLlm(response=json_text)))
    )


def upload(filename: str, file_bytes: bytes) -> dict[str, tuple[str, bytes, str]]:
    return {"file": (filename, file_bytes, "application/pdf")}


print(f"sample PDF: {len(_SAMPLE_PDF):,} bytes")


sample PDF: 14,695,023 bytes


## 3 — Happy path: ingest, get accepted, Stage 3 runs automatically

`TestClient` runs FastAPI's background tasks synchronously as part of the call, and
`PipelineService._run()` chains straight into `_run_enrichment()` once the job is
accepted -- so by the time `client.post(...)` returns below, Stage 1/2 *and* Stage 3
have both already finished, no extra waiting needed.

In [3]:
set_legitimacy(is_legitimate=True)
set_entity_response(_VALID_ENTITY_RESPONSE)

response = client.post(
    "/pipeline/ingest", files=upload("ordenanza_6801_1999.pdf", _SAMPLE_PDF), headers=auth_headers
)
print(f"status: {response.status_code}")
accepted_job_id = response.json()["jobId"]
print(f"job_id: {accepted_job_id}")

# Native Depends(get_session) commits per-request, so a fresh session_factory
# connection already sees the write -- no shutdown_resources() needed (see
# pipeline_end_to_end.ipynb section 9's note on the same point).
async def _find_job(job_id: str) -> Job | None:
    async with session_factory() as session:
        result = await session.execute(select(Job).where(Job.job_id == job_id))
        return result.scalar_one_or_none()

job = await _find_job(accepted_job_id)
print(f"job status: {job.status!r}")


ggml_cuda_init: found 1 CUDA devices (Total VRAM: 8191 MiB):
  Device 0: NVIDIA RTX A4000 Laptop GPU, compute capability 8.6, VMM: yes, VRAM: 8191 MiB
llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6631.36it/s]
2026-08-18 11:44:24.942 | INFO     | classiflow.services.audit.service:record:37 - audit | job=cbcfeb94-82f7-4460-a2d0-dde038f27166 node=node1_file_reception event=passed
2026-08-18 11:44:24.949 | INFO     | classiflow.services.audit.service:record:37 - audit | job=cbcfeb94-82f7-4460-a2d0-dde038f27166 node=node2_format_validation event=passed
2026-08-18 11:44:32.186 | INFO     | classiflow.services.audit.service:record:37 - audit | job=cbcfeb94-82f7-4460-a2d0-dde038f27166 node=extraction event=passed
2026-08-18 11:44:32.507 | INFO     | classiflow.services.audit.service:record:37 - audit | job=cbcfeb94-82f7-4460-a2d0-dde038f27166 node=node3_content_validation ev

status: 202
job_id: cbcfeb94-82f7-4460-a2d0-dde038f27166
job status: 'accepted'


## 4 — Inspect the `EnrichedRecord` Stage 3 wrote

`cleaned_text` is the exact text a future Stage 5 will chunk and embed for RAG --
compare it against the raw extracted text to see the text cleaner's repeated-line
stripping and noise removal in action. `entities`/`metadata` are the JSON blobs from
the entity extractor and metadata enricher.

In [4]:
async def _find_enriched_record(job_id: str) -> EnrichedRecord | None:
    async with session_factory() as session:
        result = await session.execute(
            select(EnrichedRecord).where(EnrichedRecord.job_id == job_id)
        )
        return result.scalar_one_or_none()

record = await _find_enriched_record(accepted_job_id)
if record is None:
    print("no EnrichedRecord found -- job wasn't accepted, or enrichment failed")
else:
    print(f"cleaned_text ({len(record.cleaned_text)} chars):\n{record.cleaned_text[:600]}...\n")
    print(f"entities: {record.entities}")
    print(f"metadata: {record.metadata_}")


cleaned_text (80678 chars):
H MICIPiLIL)!.D Dé "liAl.l'.l
ROSARIO
Direrc1ó11 Ge11tral ,te Despacho
ORDENANZA
(Nº 6.801)
Honorable Concejo:
Vuestra Comisión de Gobierno, Interpretación y Acuerdos ha tomado en
consideración lo solicitado por el Sr. José Miguel Molinari, en representación de la Comisión de
Homenaje al escritor rosarino, Don Plácido Greta en el sentido que se designe una calle de nues
tra ciudad con su nombre, al cumplirse el próximo día 18 de agosto el quinto aniversario de su
fallecimiento.
Teniendo en cuenta la trayectoria del conocido artista y compartiendo los
argumentos expresados en la reseña acompaña...

entities: {'doc_type_hint': 'ordenanza', 'number': '6801', 'year': 1999, 'issuing_body': 'Concejo Municipal', 'signatories': [], 'article_count': 3}
metadata: {'source': 'manual_upload', 'filename': 'ordenanza_6801_1999.pdf', 'language': 'es', 'sha256': '6edeb07b13d30c65baf5cae0c160293a61f5fdd965b2c78a03ccfc88748fdeb6', 'stage2_extractor_used': 'markitdown'}


## 5 — Failure path: entity extraction keeps failing

Feed the mocked entity-extraction LLM something that isn't valid JSON. Stage 3
retries `max_enrichment_retries` times (`config/enrichment.yaml`, default 2 -- so 3
attempts total), then gives up: the job is moved to `"review"` with
`review_action_needed="enrichment_failed"`, and no `EnrichedRecord` is created.

In [5]:
set_legitimacy(is_legitimate=True)
set_entity_response("not valid json at all")

response = client.post(
    "/pipeline/ingest",
    files=upload("ordenanza_6801_1999_v2.pdf", _SAMPLE_PDF),
    headers=auth_headers,
)
failed_job_id = response.json()["jobId"]
print(f"job_id: {failed_job_id}")

job = await _find_job(failed_job_id)
print(f"job status              : {job.status!r}")
print(f"review_action_needed    : {job.review_action_needed!r}")
print(f"failed_at_node          : {job.failed_at_node!r}")
print(f"rejection_reason        : {job.rejection_reason!r}")

record = await _find_enriched_record(failed_job_id)
print(f"\nEnrichedRecord exists?   {record is not None}")


llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
2026-08-18 11:44:35.324 | INFO     | classiflow.services.audit.service:record:37 - audit | job=471c012a-eea1-4116-a920-e1d3e16006cc node=node1_file_reception event=passed
2026-08-18 11:44:35.333 | INFO     | classiflow.services.audit.service:record:37 - audit | job=471c012a-eea1-4116-a920-e1d3e16006cc node=node2_format_validation event=passed
2026-08-18 11:44:42.095 | INFO     | classiflow.services.audit.service:record:37 - audit | job=471c012a-eea1-4116-a920-e1d3e16006cc node=extraction event=passed
2026-08-18 11:44:42.108 | INFO     | classiflow.services.audit.service:record:37 - audit | job=471c012a-eea1-4116-a920-e1d3e16006cc node=node3_content_validation event=passed
2026-08-18 11:44:42.113 | INFO     | classiflow.services.audit.service:record:37 - audit | job=471c012a-eea1-4116-a920-e1d3e16006cc node=node4_duplicate_control event=failed


job_id: 471c012a-eea1-4116-a920-e1d3e16006cc
job status              : 'rejected'
review_action_needed    : None
failed_at_node          : 'node4_duplicate_control'
rejection_reason        : 'Exact duplicate: SHA-256 already in store'

EnrichedRecord exists?   False
